In [1]:
import os
import glob
import zipfile
import shutil

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision import transforms

from tqdm import tqdm

import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)



device: cuda


In [2]:
base_dir = "../input/dogs-vs-cats-redux-kernels-edition"
print("base_dir exists:", os.path.exists(base_dir))
print(
    "base_dir sample:",
    os.listdir(base_dir)[:10] if os.path.exists(base_dir) else "MISSING",
)



base_dir exists: True
base_dir sample: ['cat.1714.jpg', 'cat.10025.jpg', 'dog.3113.jpg', 'cat.10782.jpg', 'cat.776.jpg', '824.jpg', '456.jpg', 'dog.2753.jpg', '1528.jpg', 'cat.3041.jpg']


In [3]:
os.makedirs("../data", exist_ok=True)
print("Created ../data")



Created ../data


In [4]:
base_dir = "../input/dogs-vs-cats-redux-kernels-edition"
train_dir = "../data/train"
test_dir = "../data/test"
print("train_dir:", train_dir, "test_dir:", test_dir)




train_dir: ../data/train test_dir: ../data/test


In [5]:
def _has_flat_jpgs(path):
    return os.path.exists(path) and len(glob.glob(os.path.join(path, "*.jpg"))) > 0


def _has_class_subdirs_with_jpgs(path, class_names):
    """
    True only if path has immediate subdirs containing the required class_names
    and each such class subdir contains at least one jpg.
    """
    if not os.path.exists(path):
        return False
    for c in class_names:
        p = os.path.join(path, c)
        if not (os.path.isdir(p) and len(glob.glob(os.path.join(p, "*.jpg"))) > 0):
            return False
    return True


def _ensure_dir(p):
    os.makedirs(p, exist_ok=True)
    return p


def safe_move(src_path, dst_dir):
    """Move a file into dst_dir unless it already exists there."""
    base = os.path.basename(src_path)
    dst_path = os.path.join(dst_dir, base)
    if os.path.abspath(src_path) == os.path.abspath(dst_path):
        return
    if os.path.exists(dst_path):
        try:
            os.remove(src_path)
        except OSError:
            pass
        return
    shutil.move(src_path, dst_dir)


def _copytree_if_needed(src, dst):
    """
    Copy a directory tree from src to dst if dst doesn't already have jpgs.
    Keeps changes minimal and avoids relying on zip extraction when already present.
    """
    if not os.path.isdir(src):
        return False
    os.makedirs(dst, exist_ok=True)

    if len(glob.glob(os.path.join(dst, "**", "*.jpg"), recursive=True)) > 0:
        return True

    for root, dirs, files in os.walk(src):
        rel = os.path.relpath(root, src)
        target_root = os.path.join(dst, rel) if rel != "." else dst
        os.makedirs(target_root, exist_ok=True)
        for fn in files:
            if fn.lower().endswith(".jpg"):
                sp = os.path.join(root, fn)
                dp = os.path.join(target_root, fn)
                if not os.path.exists(dp):
                    shutil.copy2(sp, dp)
    return len(glob.glob(os.path.join(dst, "**", "*.jpg"), recursive=True)) > 0


train_dog = _ensure_dir(os.path.join(train_dir, "dog"))
train_cat = _ensure_dir(os.path.join(train_dir, "cat"))
test_unknown = _ensure_dir(os.path.join(test_dir, "unknown"))

input_train_struct = os.path.join(base_dir, "train")
input_test_struct = os.path.join(base_dir, "test")

if (
    not _has_class_subdirs_with_jpgs(train_dir, ["cat", "dog"])
) and _has_class_subdirs_with_jpgs(input_train_struct, ["cat", "dog"]):
    print("Copying structured train data from ../input to ../data ...")
    _copytree_if_needed(
        os.path.join(input_train_struct, "cat"), os.path.join(train_dir, "cat")
    )
    _copytree_if_needed(
        os.path.join(input_train_struct, "dog"), os.path.join(train_dir, "dog")
    )

if (
    not _has_class_subdirs_with_jpgs(test_dir, ["unknown"])
) and _has_class_subdirs_with_jpgs(input_test_struct, ["unknown"]):
    print("Copying structured test data from ../input to ../data ...")
    _copytree_if_needed(
        os.path.join(input_test_struct, "unknown"), os.path.join(test_dir, "unknown")
    )

need_train_extract = not _has_class_subdirs_with_jpgs(train_dir, ["cat", "dog"])
need_test_extract = not _has_class_subdirs_with_jpgs(test_dir, ["unknown"])

if need_train_extract and os.path.exists(os.path.join(base_dir, "train.zip")):
    print("Extracting train.zip to ../data ...")
    with zipfile.ZipFile(os.path.join(base_dir, "train.zip")) as train_zip:
        train_zip.extractall("../data")

if need_test_extract and os.path.exists(os.path.join(base_dir, "test.zip")):
    print("Extracting test.zip to ../data ...")
    with zipfile.ZipFile(os.path.join(base_dir, "test.zip")) as test_zip:
        test_zip.extractall("../data")

if not _has_class_subdirs_with_jpgs(train_dir, ["cat", "dog"]):
    candidate_train_flat_dirs = [
        train_dir,
        os.path.join(train_dir, "train"),
        os.path.join("../data", "train"),
        os.path.join(base_dir, "train"),
        os.path.join(base_dir, "train", "train"),
        os.path.join("../input", "train"),
        os.path.join("../input", "dogs-vs-cats-redux-kernels-edition", "train"),
    ]
    for d in candidate_train_flat_dirs:
        if os.path.exists(d) and _has_flat_jpgs(d):
            for fp in glob.glob(os.path.join(d, "dog*.jpg")):
                safe_move(fp, train_dog)
            for fp in glob.glob(os.path.join(d, "cat*.jpg")):
                safe_move(fp, train_cat)

if not _has_class_subdirs_with_jpgs(test_dir, ["unknown"]):
    candidate_test_flat_dirs = [
        test_dir,
        os.path.join(test_dir, "test"),
        os.path.join("../data", "test"),
        os.path.join(base_dir, "test"),
        os.path.join(base_dir, "test", "test"),
        os.path.join("../input", "test"),
        os.path.join("../input", "dogs-vs-cats-redux-kernels-edition", "test"),
    ]
    for d in candidate_test_flat_dirs:
        if os.path.exists(d) and _has_flat_jpgs(d):
            for fp in glob.glob(os.path.join(d, "*.jpg")):
                safe_move(fp, test_unknown)

print("train/dog:", len(glob.glob(os.path.join(train_dog, "*.jpg"))))
print("train/cat:", len(glob.glob(os.path.join(train_cat, "*.jpg"))))
print("test/unknown:", len(glob.glob(os.path.join(test_unknown, "*.jpg"))))

if not _has_class_subdirs_with_jpgs(train_dir, ["cat", "dog"]):
    raise FileNotFoundError(
        f"Train data not found in expected structure under {train_dir}. "
        f"Expected {train_dir}/cat/*.jpg and {train_dir}/dog/*.jpg."
    )

if not _has_class_subdirs_with_jpgs(test_dir, ["unknown"]):
    raise FileNotFoundError(
        f"Test data not found in expected structure under {test_dir}. "
        f"Expected {test_dir}/unknown/*.jpg."
    )



train/dog: 11258
train/cat: 11242
test/unknown: 2500


In [6]:
print(
    "train_dir list:",
    os.listdir(train_dir)[:10] if os.path.exists(train_dir) else "MISSING",
)
print(
    "test_dir list:",
    os.listdir(test_dir)[:10] if os.path.exists(test_dir) else "MISSING",
)



train_dir list: ['dog', 'train', 'cat']
test_dir list: ['test', 'unknown']


In [7]:
print("train flat jpgs:", len(glob.glob(os.path.join(train_dir, "*.jpg"))))
print(
    "train/train flat jpgs:", len(glob.glob(os.path.join(train_dir, "train", "*.jpg")))
)
print("train/cat jpgs:", len(glob.glob(os.path.join(train_dir, "cat", "*.jpg"))))
print("train/dog jpgs:", len(glob.glob(os.path.join(train_dir, "dog", "*.jpg"))))

print("test flat jpgs:", len(glob.glob(os.path.join(test_dir, "*.jpg"))))
print("test/test flat jpgs:", len(glob.glob(os.path.join(test_dir, "test", "*.jpg"))))
print("test/unknown jpgs:", len(glob.glob(os.path.join(test_dir, "unknown", "*.jpg"))))



train flat jpgs: 0
train/train flat jpgs: 0
train/cat jpgs: 11242
train/dog jpgs: 11258
test flat jpgs: 0
test/test flat jpgs: 0
test/unknown jpgs: 2500


In [8]:
try:
    weights = torchvision.models.ResNet50_Weights.DEFAULT
    model = torchvision.models.resnet50(weights=weights)
except Exception:
    model = torchvision.models.resnet50(pretrained=True)

model.fc = torch.nn.Linear(model.fc.in_features, 2)
model = model.to(device)




Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:01<00:00, 79.2MB/s]


In [9]:
def setup_center_crop_transform():
    return transforms.Compose(
        [
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )




In [10]:
def get_labels(dataset):
    if isinstance(dataset, torch.utils.data.Subset):
        return get_labels(dataset.dataset)[dataset.indices]
    else:
        return np.array([img[1] for img in dataset.imgs])


from sklearn.model_selection import StratifiedShuffleSplit


def setup_train_val_split(labels, dryrun=False, seed=0):
    x = np.arange(len(labels))
    y = np.array(labels)
    splitter = StratifiedShuffleSplit(n_splits=1, train_size=0.8, random_state=seed)
    train_indices, val_indices = next(splitter.split(x, y))

    if dryrun:
        train_indices = np.random.choice(train_indices, 100, replace=False)
        val_indices = np.random.choice(val_indices, 100, replace=False)

    return train_indices, val_indices


def _resolve_imagefolder_root(root_dir, required_subdirs):
    """
    Bugfix: avoid selecting a parent directory that contains an extra nested folder
    like 'train' or 'test', which torchvision would interpret as a class and then fail.
    Prefer the shallowest directory whose immediate subdirs exactly match required_subdirs.
    """
    root_dir = os.path.abspath(root_dir)
    required_subdirs = list(required_subdirs)

    def immediate_subdirs(p):
        try:
            return sorted(
                [d for d in os.listdir(p) if os.path.isdir(os.path.join(p, d))]
            )
        except FileNotFoundError:
            return []

    def jpg_count_in_dir(p):
        return len(glob.glob(os.path.join(p, "*.jpg")))

    def required_jpg_total(p):
        return sum(jpg_count_in_dir(os.path.join(p, c)) for c in required_subdirs)

    def is_valid(p):
        imm = immediate_subdirs(p)
        return set(required_subdirs).issubset(set(imm)) and required_jpg_total(p) > 0

    # If root_dir is already an exact class-root, take it.
    imm = immediate_subdirs(root_dir)
    if set(imm) == set(required_subdirs) and required_jpg_total(root_dir) > 0:
        return root_dir

    # Otherwise, search under root_dir and choose the best candidate:
    # exact match of immediate subdirs, then most images, then shallowest depth.
    candidates = []
    root_depth = root_dir.count(os.sep)
    for cur, dirs, files in os.walk(root_dir):
        if is_valid(cur):
            imm = immediate_subdirs(cur)
            exact = int(set(imm) == set(required_subdirs))
            img_total = required_jpg_total(cur)
            depth = cur.count(os.sep) - root_depth
            candidates.append((exact, img_total, -depth, cur))

    if not candidates:
        raise FileNotFoundError(
            f"Could not find ImageFolder root under {root_dir} with subdirs={required_subdirs} containing images. "
            f"root_dir listing: {os.listdir(root_dir) if os.path.exists(root_dir) else 'MISSING'}"
        )

    candidates.sort(reverse=True)
    chosen = candidates[0][3]
    print(f"Resolved ImageFolder root for {required_subdirs}: {chosen}")
    return chosen


def setup_train_val_datasets(data_dir, dryrun=False):
    # Fix: Use the known class-root directly when it has cat/ and dog/ to avoid ImageFolder
    # mistakenly treating an extra nested 'train' directory as a class.
    if _has_class_subdirs_with_jpgs(data_dir, ["cat", "dog"]):
        data_root = os.path.abspath(data_dir)
    else:
        data_root = _resolve_imagefolder_root(data_dir, ["cat", "dog"])

    dataset = torchvision.datasets.ImageFolder(
        data_root,
        transform=setup_center_crop_transform(),
    )
    labels = get_labels(dataset)
    train_indices, val_indices = setup_train_val_split(labels, dryrun)

    train_dataset = torch.utils.data.Subset(dataset, train_indices)
    val_dataset = torch.utils.data.Subset(dataset, val_indices)

    return train_dataset, val_dataset


def setup_train_val_loaders(data_dir, batch_size, dryrun=False):
    train_dataset, val_dataset = setup_train_val_datasets(data_dir, dryrun=dryrun)
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=batch_size,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )
    return train_loader, val_loader




In [11]:
train_loader, val_loader = setup_train_val_loaders(
    "../data/train", batch_size=50, dryrun=False
)
print(
    len(train_loader.dataset),
    len(val_loader.dataset),
    train_loader.dataset.dataset.classes,
)




FileNotFoundError: Found no valid file for the classes train. Supported extensions are: .jpg, .jpeg, .png, .ppm, .bmp, .pgm, .tif, .tiff, .webp

In [12]:
def train_1epoch(model, train_loader, lossfun, optimizer):
    model.train()
    total_loss, total_acc = 0.0, 0.0

    for x, y in tqdm(train_loader, leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        out = model(x)
        loss = lossfun(out, y)
        _, pred = torch.max(out.detach(), 1)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_acc += torch.sum(pred == y).item()

    avg_loss = total_loss / len(train_loader.dataset)
    avg_acc = total_acc / len(train_loader.dataset)
    return avg_acc, avg_loss


def validate_1epoch(model, val_loader, lossfun):
    model.eval()
    total_loss, total_acc = 0.0, 0.0

    with torch.no_grad():
        for x, y in tqdm(val_loader, leave=False):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            out = model(x)
            loss = lossfun(out, y)
            _, pred = torch.max(out, 1)

            total_loss += loss.item() * x.size(0)
            total_acc += torch.sum(pred == y).item()

    avg_loss = total_loss / len(val_loader.dataset)
    avg_acc = total_acc / len(val_loader.dataset)
    return avg_acc, avg_loss


def train(model, optimizer, train_loader, val_loader, n_epochs):
    lossfun = torch.nn.CrossEntropyLoss()

    for epoch in tqdm(range(n_epochs)):
        train_acc, train_loss = train_1epoch(model, train_loader, lossfun, optimizer)
        val_acc, val_loss = validate_1epoch(model, val_loader, lossfun)
        print(
            f"epoch={epoch}, train loss={train_loss:.5f}, train accuracy={train_acc:.5f}, "
            f"val loss={val_loss:.5f}, val accuracy={val_acc:.5f}"
        )




In [13]:
train(
    model, optim.SGD(model.parameters(), lr=0.01), train_loader, val_loader, n_epochs=1
)




NameError: name 'train_loader' is not defined

In [14]:
def setup_test_loader(data_dir, batch_size, dryrun):
    # Fix: Use exact class-root if present (data_dir/unknown/*.jpg) to avoid ImageFolder
    # accidentally seeing an extra nested 'test' folder as a class.
    if _has_class_subdirs_with_jpgs(data_dir, ["unknown"]):
        data_root = os.path.abspath(data_dir)
    else:
        data_root = _resolve_imagefolder_root(data_dir, ["unknown"])

    dataset = torchvision.datasets.ImageFolder(
        data_root, transform=setup_center_crop_transform()
    )
    image_ids = [
        os.path.splitext(os.path.basename(path))[0] for path, _ in dataset.imgs
    ]

    if dryrun:
        dataset = torch.utils.data.Subset(dataset, range(0, 100))
        image_ids = image_ids[:100]

    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )
    return loader, image_ids


test_loader, image_ids = setup_test_loader("../data/test", batch_size=50, dryrun=False)
print(len(image_ids), image_ids[:5])




FileNotFoundError: Found no valid file for the classes test. Supported extensions are: .jpg, .jpeg, .png, .ppm, .bmp, .pgm, .tif, .tiff, .webp

In [15]:
def predict(model, loader, dog_index: int):
    pred_fun = nn.Softmax(dim=1)
    preds = []
    model.eval()
    for x, _ in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        with torch.no_grad():
            y = pred_fun(model(x))
        y = y.detach().cpu().numpy()
        y = y[:, dog_index]
        preds.append(y)
    preds = np.concatenate(preds)
    return preds


train_classes = train_loader.dataset.dataset.classes  # underlying ImageFolder
dog_index = train_classes.index("dog")

preds = predict(model, test_loader, dog_index=dog_index)
print(preds.shape, float(preds.min()), float(preds.max()), train_classes, dog_index)



NameError: name 'train_loader' is not defined

In [16]:
sample_path_candidates = [
    "../input/dogs-vs-cats-redux-kernels-edition/sample_submission.csv",
    "../input/sample_submission.csv",
    "../data/dogs-vs-cats-redux-kernels-edition/sample_submission.csv",
    "../data/sample_submission.csv",
]
sample_path = next((p for p in sample_path_candidates if os.path.exists(p)), None)
if sample_path is None:
    raise FileNotFoundError(
        "Could not find sample_submission.csv in expected locations."
    )

sample = pd.read_csv(sample_path)
sample_ids = sample["id"].astype(str).tolist()

pred_map = {str(i): float(p) for i, p in zip(image_ids, preds)}
labels = np.array([pred_map.get(str(i), 0.5) for i in sample_ids], dtype=np.float64)

eps = 1e-7
labels = np.clip(labels, eps, 1 - eps)

submission_path = "/kaggle/working/submission.csv"
sub = pd.DataFrame({"id": sample["id"].astype(int).values, "label": labels})
sub.to_csv(submission_path, index=False)
print("Wrote:", submission_path, "rows:", len(sub))
print(sub.head())
print("Submission columns:", list(sub.columns))
print("id sorted:", bool(np.all(sub["id"].values[:-1] <= sub["id"].values[1:])))

NameError: name 'image_ids' is not defined